<a href="https://colab.research.google.com/github/IgnacioAntonio82/DesarrolloDeSistemas/blob/main/Stramlit_DesarrolloSistemas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Aplicación Streamlit – Clasificador de flores Iris

#Instalar Dependencias

In [18]:
!pip install -q streamlit joblib scikit-learn

### Conectar con google drive

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


 Verificar el ZIP y extraer el modelo

In [20]:
import os
import zipfile
import shutil

# Ruta del ZIP dentro de Google Drive
ruta_zip = "/content/drive/MyDrive/DesarrolloSistemas/modelo_exportado.zip"

# Carpeta temporal donde se extraerá el contenido
carpeta_extraccion = "/content/modelo_extraido"
carpeta_modelo = "/content/modelo_exportado"

if not os.path.isfile(ruta_zip):
    raise FileNotFoundError(
        f"No se encontró el ZIP en: {ruta_zip}\n"
        "Verificá el nombre del archivo y que esté dentro de Mi unidad/DesarrolloSistemas."
    )

# Limpiar extracciones anteriores
shutil.rmtree(carpeta_extraccion, ignore_errors=True)
os.makedirs(carpeta_extraccion, exist_ok=True)

# Mostrar el contenido del ZIP y extraerlo
with zipfile.ZipFile(ruta_zip, "r") as zip_ref:
    print("Archivos incluidos en el ZIP:")
    for nombre in zip_ref.namelist():
        print(" -", nombre)
    zip_ref.extractall(carpeta_extraccion)

# Buscar los dos archivos necesarios dentro de cualquier subcarpeta
ruta_archivo_modelo = None
ruta_archivo_scaler = None

for raiz, carpetas, archivos in os.walk(carpeta_extraccion):
    if "modelo_iris.keras" in archivos:
        ruta_archivo_modelo = os.path.join(raiz, "modelo_iris.keras")
    if "scaler.pkl" in archivos:
        ruta_archivo_scaler = os.path.join(raiz, "scaler.pkl")

if ruta_archivo_modelo is None:
    raise FileNotFoundError("El ZIP no contiene un archivo llamado modelo_iris.keras.")

if ruta_archivo_scaler is None:
    raise FileNotFoundError("El ZIP no contiene un archivo llamado scaler.pkl.")

# Copiar ambos archivos a una carpeta con rutas conocidas para Streamlit
os.makedirs(carpeta_modelo, exist_ok=True)
shutil.copy2(ruta_archivo_modelo, os.path.join(carpeta_modelo, "modelo_iris.keras"))
shutil.copy2(ruta_archivo_scaler, os.path.join(carpeta_modelo, "scaler.pkl"))

print("\nModelo y scaler preparados:")
print(os.listdir(carpeta_modelo))

Archivos incluidos en el ZIP:
 - scaler.pkl
 - modelo_iris.keras
 - metadata.json

Modelo y scaler preparados:
['modelo_iris.keras', 'scaler.pkl']


##Crear la aplicación Streamlit (app.py)

##Iniciar Streamlit

In [21]:
%%writefile /content/app.py

import os
import streamlit as st
import tensorflow as tf
import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris

# --------------------------------------------------
# CONFIGURACIÓN
# --------------------------------------------------

st.set_page_config(
    page_title="Clasificador Iris",
    page_icon="🌸",
    layout="wide"
)

st.title("🌸 Clasificador de flores Iris")

st.write(
    "Aplicación de Machine Learning para clasificar "
    "flores mediante una red neuronal."
)

RUTA_MODELO = "/content/modelo_exportado/modelo_iris.keras"
RUTA_SCALER = "/content/modelo_exportado/scaler.pkl"

ESPECIES = ["Setosa", "Versicolor", "Virginica"]

COLUMNAS = [
    "Longitud sépalo",
    "Ancho sépalo",
    "Longitud pétalo",
    "Ancho pétalo"
]


# --------------------------------------------------
# CARGAR MODELO Y SCALER
# --------------------------------------------------

@st.cache_resource
def cargar_modelo():

    if not os.path.isfile(RUTA_MODELO):
        raise FileNotFoundError(
            f"No se encontró el modelo: {RUTA_MODELO}"
        )

    if not os.path.isfile(RUTA_SCALER):
        raise FileNotFoundError(
            f"No se encontró el scaler: {RUTA_SCALER}"
        )

    modelo = tf.keras.models.load_model(RUTA_MODELO)
    scaler = joblib.load(RUTA_SCALER)

    return modelo, scaler


try:

    modelo, scaler = cargar_modelo()

    st.success("Modelo y scaler cargados correctamente.")

except Exception as error:

    st.error(f"Error al cargar el modelo: {error}")
    st.stop()


# --------------------------------------------------
# FUNCIÓN DE CLASIFICACIÓN
# --------------------------------------------------

def clasificar(datos):

    # Aplicar el escalado
    datos_escalados = scaler.transform(datos)

    # Obtener probabilidades
    predicciones = modelo.predict(
        datos_escalados,
        verbose=0
    )

    if predicciones.shape[1] != 3:
        raise ValueError(
            "El modelo debe devolver tres probabilidades."
        )

    # Obtener clase de mayor probabilidad
    clases = np.argmax(predicciones, axis=1)

    return predicciones, clases


# --------------------------------------------------
# PESTAÑAS
# --------------------------------------------------

tab1, tab2 = st.tabs([
    "🌸 Clasificación individual",
    "📊 Tabla de pruebas"
])


# ==================================================
# PESTAÑA 1: CLASIFICACIÓN INDIVIDUAL
# ==================================================

with tab1:

    st.subheader("Características de la flor")

    col1, col2 = st.columns(2)

    with col1:

        sl = st.number_input(
            "Longitud del sépalo (cm)",
            min_value=0.0,
            max_value=10.0,
            value=5.1,
            step=0.1
        )

        sw = st.number_input(
            "Ancho del sépalo (cm)",
            min_value=0.0,
            max_value=10.0,
            value=3.5,
            step=0.1
        )

    with col2:

        pl = st.number_input(
            "Longitud del pétalo (cm)",
            min_value=0.0,
            max_value=10.0,
            value=1.4,
            step=0.1
        )

        pw = st.number_input(
            "Ancho del pétalo (cm)",
            min_value=0.0,
            max_value=10.0,
            value=0.2,
            step=0.1
        )

    if st.button("Clasificar flor", type="primary"):

        try:

            datos = np.array([
                [sl, sw, pl, pw]
            ])

            probabilidades, clases = clasificar(datos)

            clase = int(clases[0])

            st.success(
                f"Especie predicha: {ESPECIES[clase]}"
            )

            st.subheader("Probabilidad por especie")

            for especie, probabilidad in zip(
                ESPECIES,
                probabilidades[0]
            ):

                porcentaje = float(probabilidad) * 100

                st.write(
                    f"**{especie}: {porcentaje:.2f}%**"
                )

                st.progress(
                    float(np.clip(probabilidad, 0, 1))
                )

        except Exception as error:

            st.error(f"Error al clasificar: {error}")


# ==================================================
# PESTAÑA 2: TABLA DE PRUEBAS
# ==================================================

with tab2:

    st.subheader("Pruebas de clasificación")

    st.write(
        "Podés modificar las medidas de las flores "
        "y ejecutar varias predicciones simultáneamente."
    )

    # ----------------------------------------------
    # CARGAR DATOS DE EJEMPLO
    # ----------------------------------------------

    iris = load_iris()

    indices = [
        0, 10, 20,
        50, 60, 70,
        100, 110, 120
    ]

    if "tabla_pruebas" not in st.session_state:

        tabla_inicial = pd.DataFrame(
            iris.data[indices],
            columns=COLUMNAS
        )

        tabla_inicial["Especie real"] = [
            ESPECIES[int(iris.target[i])]
            for i in indices
        ]

        st.session_state.tabla_pruebas = tabla_inicial

    st.caption(
        "Se cargan nueve ejemplos del dataset Iris: "
        "tres de cada especie."
    )

    # ----------------------------------------------
    # EDITAR TABLA
    # ----------------------------------------------

    tabla_editada = st.data_editor(
        st.session_state.tabla_pruebas,
        use_container_width=True,
        num_rows="dynamic",
        disabled=["Especie real"],
        key="editor_iris"
    )

    st.caption(
        "Podés modificar las cuatro medidas o agregar "
        "nuevas filas. La especie real de los ejemplos "
        "originales se mantiene como referencia."
    )

    # ----------------------------------------------
    # BOTÓN PARA CLASIFICAR TODA LA TABLA
    # ----------------------------------------------

    if st.button(
        "Clasificar todas las flores",
        type="primary"
    ):

        try:

            datos = tabla_editada[COLUMNAS].apply(
                pd.to_numeric,
                errors="coerce"
            )

            if datos.empty:
                st.warning("No hay flores para clasificar.")

            elif datos.isnull().any().any():
                st.error(
                    "Hay valores vacíos o no numéricos."
                )

            elif (datos < 0).any().any():
                st.error(
                    "Las medidas no pueden ser negativas."
                )

            elif (datos > 10).any().any():
                st.error(
                    "Las medidas no pueden superar 10 cm."
                )

            else:

                # Ejecutar predicciones en lote
                probabilidades, clases = clasificar(
                    datos.to_numpy(dtype=float)
                )

                # Crear tabla de resultados
                resultados = tabla_editada.copy()

                resultados["Especie predicha"] = [
                    ESPECIES[int(c)]
                    for c in clases
                ]

                resultados["Setosa (%)"] = (
                    probabilidades[:, 0] * 100
                ).round(2)

                resultados["Versicolor (%)"] = (
                    probabilidades[:, 1] * 100
                ).round(2)

                resultados["Virginica (%)"] = (
                    probabilidades[:, 2] * 100
                ).round(2)

                # Comparar con especie real cuando existe
                resultados["Acierto"] = [
                    "Sí" if real == pred else "No"
                    for real, pred in zip(
                        resultados["Especie real"],
                        resultados["Especie predicha"]
                    )
                ]

                # Mostrar resultados
                st.subheader("Resultados de las predicciones")

                st.dataframe(
                    resultados,
                    use_container_width=True,
                    hide_index=True
                )

                # ----------------------------------
                # RESUMEN
                # ----------------------------------

                total = len(resultados)

                aciertos = (
                    resultados["Acierto"] == "Sí"
                ).sum()

                st.subheader("Resumen de pruebas")

                c1, c2, c3 = st.columns(3)

                c1.metric(
                    "Flores evaluadas",
                    total
                )

                c2.metric(
                    "Predicciones correctas",
                    int(aciertos)
                )

                c3.metric(
                    "Acierto (%)",
                    f"{aciertos / total * 100:.2f}%"
                )

                # ----------------------------------
                # DESCARGAR RESULTADOS
                # ----------------------------------

                csv = resultados.to_csv(
                    index=False
                ).encode("utf-8-sig")

                st.download_button(
                    "Descargar resultados CSV",
                    data=csv,
                    file_name="resultados_iris.csv",
                    mime="text/csv"
                )

        except Exception as error:

            st.error(
                f"Error durante las pruebas: {error}"
            )


Overwriting /content/app.py


In [22]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared.deb

!dpkg -i /content/cloudflared.deb

!cloudflared --version

(Reading database ... 127328 files and directories currently installed.)
Preparing to unpack /content/cloudflared.deb ...
Unpacking cloudflared (2026.10.7) over (2026.10.7) ...
Setting up cloudflared (2026.10.7) ...
Processing triggers for man-db (2.12.0-4build2) ...
cloudflared version 2026.10.7 (built 2026-10-30-17:05 UTC)


## Comprobar que Streamlit inició

In [23]:
import subprocess
import time
import os

# Liberar el puerto 8501 si está ocupado
subprocess.run(
    ["fuser", "-k", "8501/tcp"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Iniciar Streamlit
with open("/content/streamlit.log", "w") as log:

    proceso = subprocess.Popen(
        [
            "streamlit",
            "run",
            "/content/app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true"
        ],
        stdout=log,
        stderr=subprocess.STDOUT
    )

# Esperar a que inicie
time.sleep(5)

# Verificar que el proceso siga activo
if proceso.poll() is None:

    print("Streamlit iniciado correctamente.")
    print("Puerto: 8501")

else:

    print("Error al iniciar Streamlit.")
    with open("/content/streamlit.log") as log:
        print(log.read())

Streamlit iniciado correctamente.
Puerto: 8501


##Crear el acceso público con LocalTunnel

In [24]:
!cloudflared tunnel --no-autoupdate --url http://127.0.0.1:8501

2026-11-06T01:31:53Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-06T01:31:53Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-06T01:31:58Z INF +--------------------------------------------------------------------------------------------+
2026-11-06T01:31:58Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-06T01:31:58Z INF |  https://roses-robinson-ppc-justice.trycloudflare.com 